In [7]:
%pip install opencv-python

Note: you may need to restart the kernel to use updated packages.


In [ ]:
import cv2
import numpy as np
print("OpenCV Version:", cv2.__version__)

ModuleNotFoundError: No module named 'matplotlib'

In [ ]:
%pip install sounddevice

  Using cached sounddevice-0.5.6-py3-none-win_amd64.whl.metadata (1.4 kB)
  Using cached cffi-2.1.1-cp311-cp311-win_amd64.whl.metadata (2.6 kB)
  Using cached pycparser-3.0-py3-none-any.whl.metadata (8.2 kB)
Using cached sounddevice-0.5.6-py3-none-win_amd64.whl (1.0 MB)
Using cached cffi-2.1.1-cp311-cp311-win_amd64.whl (185 kB)
Using cached pycparser-3.0-py3-none-any.whl (48 kB)
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 26.2.1
[notice] To update, run: C:\Users\patel\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [ ]:
%pip install tensorflow==2.15.1

Note: you may need to restart the kernel to use updated packages.Collecting tensorflow==2.15.1
  Using cached tensorflow-2.15.1-cp311-cp311-win_amd64.whl.metadata (3.4 kB)
  Using cached tensorflow_intel-2.15.1-cp311-cp311-win_amd64.whl.metadata (4.9 kB)
  Using cached absl_py-2.5.0-py3-none-any.whl.metadata (3.3 kB)
  Using cached astunparse-1.6.3-py2.py3-none-any.whl.metadata (4.4 kB)
  Using cached flatbuffers-25.12.19-py2.py3-none-any.whl.metadata (1.0 kB)
  Using cached gast-0.7.0-py3-none-any.whl.metadata (1.5 kB)
  Using cached google_pasta-0.2.0-py3-none-any.whl.metadata (814 bytes)
  Using cached h5py-3.16.0-cp311-cp311-win_amd64.whl.metadata (3.1 kB)
  Using cached libclang-18.1.1-py2.py3-none-win_amd64.whl.metadata (5.3 kB)
  Using cached ml_dtypes-0.3.2-cp311-cp311-win_amd64.whl.metadata (20 kB)
  Using cached numpy-1.26.4-cp311-cp311-win_amd64.whl.metadata (61 kB)
  Using cached opt_einsum-3.4.0-py3-none-any.whl.metadata (6.3 kB)
  Using cached protobuf-4.25.9-cp310-abi3-w

  You can safely remove it manually.
  You can safely remove it manually.
ERROR: Could not install packages due to an OSError: [Errno 2] No such file or directory: 'C:\\Users\\patel\\AppData\\Local\\Packages\\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\\LocalCache\\local-packages\\Python311\\site-packages\\tensorflow\\include\\external\\com_github_grpc_grpc\\src\\core\\ext\\filters\\client_channel\\lb_policy\\grpclb\\client_load_reporting_filter.h'
HINT: This error might have occurred since this system does not have Windows Long Path support enabled. You can find information on how to enable this at https://pip.pypa.io/warnings/enable-long-paths


[notice] A new release of pip is available: 24.0 -> 26.2.1
[notice] To update, run: C:\Users\patel\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [ ]:
import cv2
import numpy as np
import sounddevice as sd # type: ignore
import tensorflow as tf
import time

from pathlib import Path

PROJECT_DIR = Path(r"C:\Users\patel\OneDrive\Desktop\R&W\Deep Learning\DL_PR4B")

MODEL_PATH = PROJECT_DIR / "models" / "audio" / "soundclassifier_with_metadata.tflite"
LABELS_PATH = PROJECT_DIR / "models" / "audio" / "labels.txt"

print("Model exists:", MODEL_PATH.exists())
print("Labels exists:", LABELS_PATH.exists())

# Audio settings from the exported model
SAMPLE_RATE = 44032
DURATION = 1

# Load labels
with open(LABELS_PATH, "r", encoding="utf-8") as f:
    labels = [line.strip() for line in f.readlines()]

# Load TFLite model
interpreter = tf.lite.Interpreter(model_path=str(MODEL_PATH))
interpreter.allocate_tensors()

input_details = interpreter.get_input_details()
output_details = interpreter.get_output_details()

print("===================================")
print("     B2 SMART CLASSROOM SOUND AI")
print("===================================")
print("Classes:", labels)
print("Model loaded successfully! ✅")
print("\nStarting microphone...")
print("Make a sound: Clap, Talk, Keyboard, Bell, or Background Noise.")
print("Press Q to quit.\n")

while True:

    # Record 1 second of audio
    audio = sd.rec(
        int(SAMPLE_RATE * DURATION),
        samplerate=SAMPLE_RATE,
        channels=1,
        dtype="float32"
    )

    sd.wait()

    # Convert to required shape
    audio = audio.flatten()

    # Make sure exactly 44032 samples are provided
    if len(audio) < 44032:
        audio = np.pad(audio, (0, 44032 - len(audio)))

    audio = audio[:44032]

    # Prepare model input
    input_data = np.expand_dims(audio, axis=0).astype(np.float32)

    # Run prediction
    interpreter.set_tensor(
        input_details[0]["index"],
        input_data
    )

    interpreter.invoke()

    prediction = interpreter.get_tensor(
        output_details[0]["index"]
    )[0]

    # Get highest probability
    index = np.argmax(prediction)
    confidence = float(prediction[index])

    label = labels[index] if index < len(labels) else f"Class {index}"

    print(
        f"Prediction: {label} | "
        f"Confidence: {confidence * 100:.2f}%"
    )

    # Simple display window
    frame = np.zeros((300, 800, 3), dtype=np.uint8)

    text = f"{label}: {confidence * 100:.1f}%"

    cv2.putText(
        frame,
        "B2 Smart Classroom Sound AI",
        (40, 70),
        cv2.FONT_HERSHEY_SIMPLEX,
        1,
        (255, 255, 255),
        2
    )

    cv2.putText(
        frame,
        text,
        (40, 160),
        cv2.FONT_HERSHEY_SIMPLEX,
        1.2,
        (0, 255, 0),
        3
    )

    cv2.putText(
        frame,
        "Press Q to quit",
        (40, 240),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.7,
        (200, 200, 200),
        2
    )

    cv2.imshow("B2 Audio Classifier", frame)

    if cv2.waitKey(1) & 0xFF == ord("q"):
        break

cv2.destroyAllWindows()

Model exists: True
Labels exists: True
     B2 SMART CLASSROOM SOUND AI
Classes: ['0  keyboard', '1 Background Noise', '2 bell', '3 clap', '4 talk']
Model loaded successfully! ✅

Starting microphone...
Make a sound: Clap, Talk, Keyboard, Bell, or Background Noise.
Press Q to quit.

Prediction: 2 bell | Confidence: 60.62%
Prediction: 4 talk | Confidence: 60.84%
Prediction: 2 bell | Confidence: 66.10%
Prediction: 4 talk | Confidence: 51.15%
Prediction: 4 talk | Confidence: 61.04%
Prediction: 4 talk | Confidence: 94.03%
Prediction: 2 bell | Confidence: 66.08%
Prediction: 4 talk | Confidence: 90.53%
Prediction: 0  keyboard | Confidence: 36.25%
Prediction: 2 bell | Confidence: 39.21%
Prediction: 4 talk | Confidence: 82.46%
Prediction: 1 Background Noise | Confidence: 36.41%
Prediction: 2 bell | Confidence: 53.74%
Prediction: 1 Background Noise | Confidence: 32.10%
Prediction: 0  keyboard | Confidence: 50.79%
Prediction: 0  keyboard | Confidence: 72.07%
Prediction: 4 talk | Confidence: 97.54